In [2]:
import pandas as pd

In [8]:
df = pd.read_excel("horarios_reuniao.xlsx")

In [9]:
df.head()

,Carimbo de data/hora,Endereço de e-mail,Discente ou Docente?,Disponibilidade de horários [Segunda-feira],Disponibilidade de horários [Terça-feira],Disponibilidade de horários [Quarta-feira],Disponibilidade de horários [Quinta-feira],Disponibilidade de horários [Sexta-feira]
0,2026-09-09 10:39:53.092,wiliane.lima@alunos.ufersa.edu.br,Discente,M45,M45,"T12, T23","T12, T23","T12, T23"
1,2026-09-09 10:42:27.326,joao.andrade30247@alunos.ufersa.edu.br,Discente,"M23, M45, T12, T23","M23, M45, T12, T23","M23, M45, T12, T23","M23, M45, T12, T23","M23, M45, T12, T23"
2,2026-09-09 10:43:11.550,frederico.ferreira@alunos.ufersa.edu.br,Discente,"T12, T23, T45",T45,"M23, M45, T45","T12, T23, T45","T12, T23, T45"
3,2026-09-09 10:44:40.659,valderi.neto@alunos.ufersa.edu.br,Discente,T45,"T45, N12","T45, N12","T45, N12","T45, N12"
4,2026-09-09 10:45:03.059,thiago.nunes@alunos.ufersa.edu.br,Discente,N12,N12,"M45, T45, N12","M23, M45, T12, N12","M23, N12"


In [10]:
# 1. Selecionar as colunas que representam os dias da semana
colunas_dias = [col for col in df.columns if 'Disponibilidade de horários' in col]

# 2. Reorganizar os dados (Melt) para que os dias da semana virem linhas
df_melted = df.melt(id_vars=['Discente ou Docente?'], 
                    value_vars=colunas_dias, 
                    var_name='Dia da Semana', 
                    value_name='Horarios')

# 3. Remover valores nulos
df_melted = df_melted.dropna(subset=['Horarios'])

# 4. Limpar o nome da coluna 'Dia da Semana'
# O uso de expand=False é CRUCIAL aqui para evitar o erro "cannot reindex..."
df_melted['Dia da Semana'] = df_melted['Dia da Semana'].str.extract(r'\[(.*?)\]', expand=False)

# 5. Como os horários estão agrupados por vírgula na mesma célula, vamos dividi-los
df_melted['Horarios'] = df_melted['Horarios'].astype(str).str.split(',')
df_exploded = df_melted.explode('Horarios')

# 6. Remover espaços em branco extras (ex: ' T23' vira 'T23')
df_exploded['Horarios'] = df_exploded['Horarios'].str.strip()

# Reseta o index por segurança para limpar as duplicações deixadas pelo explode()
df_exploded = df_exploded.reset_index(drop=True)

# 7. Criar a tabela de frequência 
tabela_freq = pd.crosstab(
    index=[df_exploded['Discente ou Docente?'], df_exploded['Horarios']],
    columns=df_exploded['Dia da Semana']
)

# 8. Reordenar as colunas para ficarem na sequência correta da semana
ordem_semana = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira', 'Sexta-feira']
colunas_presentes = [dia for dia in ordem_semana if dia in tabela_freq.columns]
tabela_freq = tabela_freq[colunas_presentes]

# Exibe o resultado final no Jupyter Notebook
display(tabela_freq)


Dia da Semana                  Segunda-feira  Terça-feira  Quarta-feira  \
Discente ou Docente? Horarios                                             
Discente             M23                  11           10             9   
                     M45                   7            9            12   
                     N12                   9           11            10   
                     T12                  10           10            17   
                     T23                  10            8            14   
                     T45                  11           10             9   
Docente              M23                   1            4             2   
                     M45                   0            3             3   
                     N12                   0            1             3   
                     T12                   2            2             3   
                     T23                   4            3             3   
                     T45                   2            2             2   

Dia da Semana                  Quinta-feira  Sexta-feira  
Discente ou Docente? Horarios                             
Discente             M23                  8            6  
                     M45                  8            5  
                     N12                 10            6  
                     T12                 20           12  
                     T23                 12            8  
                     T45                 11            6  
Docente              M23                  3            1  
                     M45                  3            1  
                     N12                  3            1  
                     T12                  3            1  
                     T23                  4            3  
                     T45                  3            2

In [14]:
# 1. Filtra as linhas onde a pessoa é Docente
filtro_docentes = df['Discente ou Docente?'] == 'Docente'

# 2. Filtra as linhas onde a disponibilidade de Quinta-feira não está vazia
# (O .notna() garante que pegaremos apenas as pessoas que marcaram algum horário)
filtro_quinta = df['Disponibilidade de horários [Quinta-feira]'].notna()
#filtro_quinta = df['Disponibilidade de horários [Quinta-feira]'].str.contains('T12', na=False)

# 3. Aplica os filtros simultaneamente usando o '&' (E) e seleciona as colunas desejadas
colunas_para_exibir = ['Endereço de e-mail', 'Disponibilidade de horários [Quinta-feira]']
docentes_quinta = df.loc[filtro_docentes & filtro_quinta, colunas_para_exibir]

# Exibe o resultado
display(docentes_quinta)


,Endereço de e-mail,Disponibilidade de horários [Quinta-feira]
22,petrucio@ufersa.edu.br,"T12, T23, T45, N12"
24,cecilio.martins@ufersa.edu.br,T23
25,engcomputacao.pdf@ufersa.edu.br,"M23, M45, T12, T23, T45"
26,kennedy.lopes@ufersa.edu.br,N12
27,walber.silva@ufersa.edu.br,"M23, M45, T12, T23, T45, N12"
29,francisco.segundo@ufersa.edu.br,"M23, M45"


In [18]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Filtrar o DataFrame original APENAS para Docentes
df_docentes = df[df['Discente ou Docente?'] == 'Docente'].copy()

# 2. Tratar os dados (Melt e Explode)
colunas_dias = [col for col in df_docentes.columns if 'Disponibilidade de horários' in col]

df_melted = df_docentes.melt(id_vars=['Endereço de e-mail'], 
                             value_vars=colunas_dias, 
                             var_name='Dia da Semana', 
                             value_name='Horarios')

df_melted = df_melted.dropna(subset=['Horarios'])
df_melted['Dia da Semana'] = df_melted['Dia da Semana'].str.extract(r'\[(.*?)\]', expand=False)
df_melted['Nome'] = df_melted['Endereço de e-mail'].str.split('@').str[0].str.replace('.', ' ', regex=False).str.title()

df_melted['Horarios'] = df_melted['Horarios'].astype(str).str.split(',')
df_exploded = df_melted.explode('Horarios')
df_exploded['Horarios'] = df_exploded['Horarios'].str.strip()
df_exploded = df_exploded.reset_index(drop=True)

# ---------------------------------------------------------
# PARTE 1: Tabela com os Nomes de quem pode em cada horário
# ---------------------------------------------------------
tabela_nomes = df_exploded.pivot_table(
    index='Horarios',
    columns='Dia da Semana',
    values='Nome',
    aggfunc=lambda x: ', '.join(x.unique()),
    fill_value='-'
)

ordem_semana = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira', 'Sexta-feira']
colunas_presentes = [dia for dia in ordem_semana if dia in tabela_nomes.columns]
tabela_nomes = tabela_nomes[colunas_presentes]

print("--- TABELA DE NOMES (DOCENTES) ---")
display(tabela_nomes)



--- TABELA DE NOMES (DOCENTES) ---


Dia da Semana,Segunda-feira,Terça-feira,Quarta-feira,Quinta-feira,Sexta-feira
Horarios,,,,,
M23,Engcomputacao Pdf,"Adller Guimaraes, Engcomputacao Pdf, Walber Si...","Adller Guimaraes, Kennedy Lopes","Engcomputacao Pdf, Walber Silva, Francisco Seg...",Engcomputacao Pdf
M45,-,"Adller Guimaraes, Walber Silva, Francisco Segundo","Adller Guimaraes, Engcomputacao Pdf, Kennedy L...","Engcomputacao Pdf, Walber Silva, Francisco Seg...",Engcomputacao Pdf
N12,-,Walber Silva,"Petrucio, Kennedy Lopes, Walber Silva","Petrucio, Kennedy Lopes, Walber Silva",Petrucio
T12,"Adller Guimaraes, Engcomputacao Pdf","Engcomputacao Pdf, Walber Silva","Petrucio, Engcomputacao Pdf, Walber Silva","Petrucio, Engcomputacao Pdf, Walber Silva",Petrucio
T23,"Adller Guimaraes, Cecilio Martins, Engcomputac...","Cecilio Martins, Engcomputacao Pdf, Walber Silva","Cecilio Martins, Engcomputacao Pdf, Walber Silva","Petrucio, Cecilio Martins, Engcomputacao Pdf, ...","Petrucio, Cecilio Martins, Francisco Segundo"
T45,"Adller Guimaraes, Francisco Segundo","Adller Guimaraes, Walber Silva","Adller Guimaraes, Walber Silva","Petrucio, Engcomputacao Pdf, Walber Silva","Petrucio, Francisco Segundo"


Discente:
- Segunda - M23 (11 alunos)
- Terça - M23 (10 alunos)
- Quarta - T12 & 3 (17 alunos)
- Quinta - T12 & 3 (20 alunos)
- Sexta - T12 (12 alunos)

Docente:
- Segunda - T23 (Melhor dia dos professores: Adller, Pedro, Cecilio, Segundo)
- Terça - M23
- Quarta - T12 & 3
- Quinta - T12 & 3
- Sexta - T23

Escolha:
- Quinta - T12 & 3 (Professores: Pedro, Cecilio, Petrucio(talvez off), Walber)